# Simonyi dome open/close retrospective (RSO-1175 / RSO-1176)

Replays the written weather guidance against EFD telemetry, minute by minute, to answer two questions:

1. **One night (default dayObs 20261028):** when would each version of the rules have allowed the dome to open, did any humidity spike reach a closing limit, and what did the logs, Watcher and scheduler record?
2. **Commissioning through operations:** per night, the hours the guidance allowed the dome to be open versus the hours it was actually open. This flags nights that may have been missed (closed but allowed) and risky nights (open past a closing limit).

Guidance replayed (Confluence OOD space):
* *Weather Constraints* (page 39696225, v36, 2026-10-27): summary table (dewpoint spread governs, humidity de-prioritized) **and** the "Deciding to open" prose (RH > 70% = do not open).
* *Recovering from precipitation* (page 1241514042, v10, draft): RH > 75% keeps the dome closed; 20-30 min of safe conditions before opening.
* A **proposed** rule: open when the rolling 30-min *median* spread is > 5 °C and the 30-min *minimum* is > 2 °C. A short spike does not restart the clock.
* A **proposed thaw rule** for precipitation (RSO-1183), in place of "no opening until an engineering inspection". It classifies each precipitation episode as rain or snow from the wet-bulb temperature, and allows opening once no freezing has occurred since, the air has stayed above a floor, enough thaw degree-hours have accumulated, and the air stays above freezing until sunrise.

Run this on the USDF RSP (Nublado) or Times Square. Data access goes through `rubin_nights`, the same as the other notebooks in this repo.
Cloud cover has no direct EFD measurement. The allskycam cloud index (rubin_allskycam products under `ALLSKY_OUT`) is used where available, plus DIMM/RINGSS measurement cadence as a lower-bound proxy for clear sky (neither has published since mid-2026); review the all-sky movie for the night by eye.

*AI-assisted (NOIRL-POL-006): drafted with Claude, to be reviewed before use.*

In [ ]:
# Parameters
DAY_OBS = 20261028  # night to replay in detail
SEASON_MIN = 20250415  # start of the season-wide scan (LSSTCam on sky)
SEASON_MAX = 20261101  # end of the season-wide scan
RUN_SEASON = False  # the season scan makes many EFD queries; turn on when wanted
SEASON_BIN = "5m"  # aggregation for the season scan
OUTSIDE_INDEX = 301  # ESS:301 = outside weather station
CAMERA_DB = "lsst.MTCamera"  # MTCamera influx database (not the EFD)
DYN_TOPIC = "lsst.MTCamera.utiltrunk_ut"
DYN_FIELD = (
    "CoolPipeSplyTemp"  # Dynalene supply measured at the camera (on the telescope)
)
DYN_OFFSET = 0.0  # added to DYN_FIELD; the camera reading is already the on-telescope temperature
# (source setpoint -2 C, warms to ~2 C at the TMA supply and ~3.5 C at the camera)
TOPEND_INDEX = (
    111  # ESS:111 = top end (dew point recorded on closure per the guidance page)
)
SITE_TZ = "America/Santiago"
# Off-sky periods excluded from the "possible missed" screen: nights when the system could not observe
# for reasons other than weather policy. dayObs ranges are inclusive. Extend from the ROSP-42 override file.
UNAVAILABLE = [
    (20260715, 20261025, "storm evacuation, road closures, planned maintenance, LSSTCam warm (PTH attribution timeline)"),
]

In [ ]:
try:
    import rubin_nights  # noqa: F401
except ImportError:
    !pip install --user --upgrade git+https://github.com/lsst-sims/rubin_nights.git --no-deps > /dev/null 2>&1

import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display
from astropy.time import Time, TimeDelta

from rubin_nights import connections
import rubin_nights.dayobs_utils as rn_dayobs
import rubin_nights.observatory_status as observatory_status

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

from rubin_nights.influx_query import InfluxQueryClient

endpoints = connections.get_clients()
efd = endpoints["efd"]

# The camera writes utiltrunk telemetry to its own influx database, reached with the same RSP token.
try:
    cam = InfluxQueryClient(
        efd.site,
        db_name=CAMERA_DB,
        repertoire_site=efd.repertoire_site,
        auth=("user", connections.get_access_token()),
    )
except Exception as e:  # noqa: BLE001
    print(f"camera influx via repertoire failed ({e}); trying segwarides")
    cam = InfluxQueryClient(efd.site, db_name=CAMERA_DB)

## Guidance thresholds

All limits are expressed as the ambient dewpoint spread (T − T_dew, °C) where possible. The page's own formula, T_dew = T − (100 − RH)/5, gives RH 70% → 6 °C, 75% → 5 °C, 80% → 4 °C and 90% → 2 °C.

In [ ]:
G = dict(
    spread_open=5.0,  # table: can open above this
    spread_close=2.0,  # table: close below this (ambient, instantaneous)
    sensor_close=2.0,  # any sensor within 2 C of its dew point ...
    sensor_close_min=20,  # ... for ~20 min -> close
    rh_prose=70.0,  # "Deciding to open" prose: RH > 70% -> do not open
    rh_precip=75.0,  # precipitation proposal: RH > 75% -> keep closed
    wind_avg_close=20.0,  # 5-min average wind, m/s
    gust_close=25.0,  # gusts, m/s (multiple points)
    reopen_stable_min=30,  # "conditions stable for 30 min"
    before_sunrise_h=1.0,  # "> 1 hour before sunrise"
    spike_rh=80.0,  # what the night report called a spike
    # precipitation handling (RSO-1183)
    crew_leave_hour=17,  # current policy proxy: precipitation ending after this local hour on the dayObs date
    # cannot be inspected by the day crew, so the dome stays closed all night
    precip_gap_min=30,  # precipitation events closer than this are one episode
    precip_lookback_h=72,  # how far back to look for the last episode and for freezing degree-hours
    rain_dry_min=30,  # rain path: minutes dry after the episode ends
    wb_rain=0.0,  # wet-bulb minimum during the episode >= this (and no snow flag) -> treat as rain
    fdh_tol=0.05,  # freezing degree-hours (C*h below 0) since the episode treated as zero
    thaw_min_air=3.0,  # snow path: air must stay >= this since the episode ended
    thaw_min_degh=10.0,  # snow path: thaw degree-hours (C*h above 0) needed since the episode ended
    # sky and observatory-state gates (season screen)
    cloud_clear=0.3,  # allskycam cloud_idx at or below this counts as clear (0.30 = thin cloud)
    cloud_window_min=30,  # rolling median window for the cloud index, like the spread rule
    cloud_min_coverage=0.5,
    # deep allskycam index (Hipparcos V<7.5; measures thin cloud on the sharp stints, FWHM ~1.4 px)
    ext_clear=0.2,  # ext_deep (grey extinction, mag) at or below this = clear; matches thin_frac in rubin_allskycam
    # deep-index transparency gates (30-min rolling median ext_deep <= limit), name -> mag
    ext_gates={"ext1": 1.0,  # proposed: open through thin / moderate cloud
               "ext02": 0.2},  # likely aligned with current guidance (clear sky only)
    sharp_stints=("8323/0552@20261016", "9118/0165@20220223"),  # FWHM ~1.4 px: ext_deep sensitive to thin cloud
    ext_bins=(0.2, 0.5, 1.0),  # thickness classes: clear / thin / moderate / thick (ext_deep saturates near 1.5-2 mag)
    cell_opaque=0.6,  # NSIDE-8 cell cloud_idx_deep at or above this counts the cell as (near) opaque  # fraction of the night with cloud data needed to screen a night
    down_types=("DOWNTIME", "FAULT"),  # observatory states that remove time from the screen
    refreeze_air=2.0,  # both paths: air must stay >= this until sunrise (hindcast stand-in for the forecast)
)
G

## Helpers

In [ ]:
def _iso(t):
    return f"{Time(t).utc.isot}Z"


def efd_agg(topic, field, t0, t1, every="1m", agg="mean", index=None, client=None):
    # Aggregated influx series (UTC index); `client` defaults to the EFD. Empty Series if the topic or field is missing.
    where = f"time >= '{_iso(t0)}' AND time <= '{_iso(t1)}'"
    if index is not None:
        where += f" AND salIndex = {int(index)}"
    q = f'SELECT {agg}("{field}") AS v FROM "{topic}" WHERE {where} GROUP BY time({every}) fill(none)'
    df = (client or efd).query(q)
    if df is None or len(df) == 0 or "v" not in df:
        return pd.Series(dtype=float, name=field)
    return df["v"].astype(float).rename(field)


def efd_raw(topic, fields, t0, t1, index=None):
    try:
        df = efd.select_time_series(topic, fields, Time(t0), Time(t1), index=index)
    except Exception as e:  # noqa: BLE001
        print(f"{topic}: {type(e).__name__}: {e}")
        return pd.DataFrame()
    return df if df is not None else pd.DataFrame()


def first_field(topic, prefix):
    # The first field of `topic` whose name starts with `prefix` (handles array fields such as temperatureItem0).
    try:
        f = efd.get_fields(topic)
    except Exception:  # noqa: BLE001
        return None
    col = "fieldKey" if "fieldKey" in f.columns else f.columns[0]
    names = sorted(n for n in f[col].astype(str) if n.startswith(prefix))
    return names[0] if names else None


def outside_weather(t0, t1, every="1m", index=OUTSIDE_INDEX):
    t_field = (
        first_field("lsst.sal.ESS.temperature", "temperatureItem") or "temperatureItem0"
    )
    cols = {
        "T": efd_agg("lsst.sal.ESS.temperature", t_field, t0, t1, every, "mean", index),
        "RH": efd_agg(
            "lsst.sal.ESS.relativeHumidity",
            "relativeHumidityItem",
            t0,
            t1,
            every,
            "mean",
            index,
        ),
        "RH_max": efd_agg(
            "lsst.sal.ESS.relativeHumidity",
            "relativeHumidityItem",
            t0,
            t1,
            every,
            "max",
            index,
        ),
        "Td": efd_agg(
            "lsst.sal.ESS.dewPoint", "dewPointItem", t0, t1, every, "mean", index
        ),
        "Td_max": efd_agg(
            "lsst.sal.ESS.dewPoint", "dewPointItem", t0, t1, every, "max", index
        ),
        "wind": efd_agg("lsst.sal.ESS.airFlow", "speed", t0, t1, every, "mean", index),
        "gust": efd_agg(
            "lsst.sal.ESS.airFlow", "maxSpeed", t0, t1, every, "max", index
        ),
    }
    w = pd.concat({k: v for k, v in cols.items()}, axis=1)
    if len(w) == 0:
        return w
    if w["Td"].isna().all() and not w["RH"].isna().all():
        # fall back to the page's approximation if no dewPoint topic for this index
        w["Td"] = w["T"] - (100 - w["RH"]) / 5
        w["Td_max"] = w["Td"]
    w["spread"] = w["T"] - w["Td"]
    # worst case within the bin: highest dew point against mean T
    w["spread_min"] = w["T"] - w["Td_max"]
    w["spread_formula"] = (100 - w["RH"]) / 5
    w["wind_avg5"] = w["wind"].rolling("5min").mean()
    return w


def precipitation(t0, t1):
    df = efd_raw(
        "lsst.sal.ESS.logevent_precipitation",
        ["sensorName", "raining", "snowing", "salIndex"],
        t0,
        t1,
    )
    if len(df) == 0:
        return df
    return df[
        (df.get("raining", False) == True) | (df.get("snowing", False) == True)
    ]  # noqa: E712


def to_local(idx):
    return idx.tz_convert(SITE_TZ)


def _ts(t):
    t = pd.Timestamp(Time(t).utc.datetime) if not isinstance(t, pd.Timestamp) else t
    return t.tz_localize("UTC") if t.tzinfo is None else t.tz_convert("UTC")


def wet_bulb(T, RH):
    # Stull (2011) wet-bulb temperature in C from air temperature (C) and RH (%); good to ~0.3 C for RH 5-99%.
    RH = np.clip(RH, 5, 99)
    return (
        T * np.arctan(0.151977 * np.sqrt(RH + 8.313659))
        + np.arctan(T + RH)
        - np.arctan(RH - 1.676331)
        + 0.00391838 * RH**1.5 * np.arctan(0.023101 * RH)
        - 4.686035
    )


def precip_episodes(t0, t1, gap_min=None, index=OUTSIDE_INDEX):
    # Precipitation episodes from ESS logevent_precipitation state changes -> DataFrame(start, end, snow_flag), UTC.
    # An episode runs from a raining/snowing True event to the next all-False event; episodes closer than gap_min merge.
    gap_min = G["precip_gap_min"] if gap_min is None else gap_min
    cols = ["start", "end", "snow_flag"]
    ev = efd_raw(
        "lsst.sal.ESS.logevent_precipitation",
        ["raining", "snowing", "salIndex"],
        t0,
        t1,
    )
    if len(ev) == 0:
        return pd.DataFrame(columns=cols)
    if "salIndex" in ev and (ev["salIndex"] == index).any():
        ev = ev[ev["salIndex"] == index]
    ev = ev.sort_index()
    rain = (
        ev["raining"].astype(bool)
        if "raining" in ev
        else pd.Series(False, index=ev.index)
    )
    snow = (
        ev["snowing"].astype(bool)
        if "snowing" in ev
        else pd.Series(False, index=ev.index)
    )
    eps, start, sflag, last_on = [], None, False, None
    for t, r_, s_ in zip(ev.index, rain, snow):
        if r_ or s_:
            if start is None:
                start, sflag = t, False
            sflag, last_on = sflag or s_, t
        elif start is not None:
            eps.append((start, t, sflag))
            start = None
    if (
        start is not None
    ):  # still on at t1, or no closing event: end 5 min after the last True
        eps.append((start, min(last_on + pd.Timedelta(minutes=5), _ts(t1)), sflag))
    merged = []
    for s, e, f in eps:
        if merged and (s - merged[-1][1]) < pd.Timedelta(minutes=gap_min):
            merged[-1] = (merged[-1][0], max(merged[-1][1], e), merged[-1][2] or f)
        else:
            merged.append((s, e, f))
    return pd.DataFrame(merged, columns=cols)


def classify_episodes(E, wx, g=G):
    # Add wet-bulb minimum and phase ('rain' or 'snow') to episodes, using the outside weather series wx (T, RH).
    E = E.copy()
    if len(E) == 0:
        return E.assign(wb_min=pd.Series(dtype=float), phase=pd.Series(dtype=str))
    wb = wet_bulb(wx["T"], wx["RH"])
    E["wb_min"] = [
        float(wb.loc[s:e].min()) if len(wb.loc[s:e]) else np.nan
        for s, e in zip(E.start, E.end)
    ]
    E["phase"] = np.where(
        E.snow_flag | ~(E.wb_min >= g["wb_rain"]), "snow", "rain"
    )  # NaN wet-bulb -> snow
    return E


def precip_gates(index, E, Tser, day_obs, sunrise0, g=G):
    # Per-sample precipitation gates on `index` (UTC):
    #   dry_current: current-policy proxy. Any episode ending after crew_leave_hour local on the dayObs date
    #                (or still going) keeps the rest of the night closed, since nobody can inspect.
    #   dry_thaw:    proposed rule. Rain: rain_dry_min minutes dry, no freezing since, no refreeze before sunrise.
    #                Snow: no freezing since, air >= thaw_min_air since, >= thaw_min_degh thaw degree-hours, no refreeze.
    out = pd.DataFrame(index=index)
    out["episode"] = -1
    for c, v in [
        ("precip_now", False),
        ("since_end_min", np.nan),
        ("phase", "none"),
        ("tdh", np.nan),
        ("fdh", np.nan),
        ("tmin_since", np.nan),
    ]:
        out[c] = v
    T = Tser.dropna().sort_index()
    if len(T):
        dt_h = (
            T.index.to_series().diff().dt.total_seconds().fillna(0).clip(upper=900)
            / 3600.0
        )  # gaps > 15 min not integrated
        cpos = (T.clip(lower=0) * dt_h).cumsum()
        cneg = ((-T).clip(lower=0) * dt_h).cumsum()
    crew_leave = pd.Timestamp(
        f"{str(day_obs)[:4]}-{str(day_obs)[4:6]}-{str(day_obs)[6:8]} {g['crew_leave_hour']:02d}:00",
        tz=SITE_TZ,
    ).tz_convert("UTC")
    lock_current = pd.Series(False, index=index)
    if len(E):
        E = E.sort_values("start").reset_index(drop=True)
        k = np.searchsorted(E.start.values, index.values, side="right") - 1
        out["episode"] = k
        for j in np.unique(k[k >= 0]):
            ep = E.loc[j]
            sel = k == j
            ts = index[sel]
            now = ts <= ep.end
            out.loc[ts, "precip_now"] = now
            out.loc[ts, "since_end_min"] = (ts - ep.end).total_seconds() / 60.0
            out.loc[ts, "phase"] = ep.phase
            if len(T):
                at = lambda c, x: float(
                    np.interp(x.value, c.index.asi8, c.values)
                )  # noqa: E731
                c0p, c0n = at(cpos, ep.end), at(cneg, ep.end)
                out.loc[ts, "tdh"] = [max(0.0, at(cpos, x) - c0p) for x in ts]
                out.loc[ts, "fdh"] = [max(0.0, at(cneg, x) - c0n) for x in ts]
                seg = T.loc[ep.end :]
                tmin = seg.cummin() if len(seg) else seg
                out.loc[ts, "tmin_since"] = (
                    tmin.reindex(ts, method="ffill").values if len(tmin) else np.nan
                )
            if ep.end > crew_leave:
                lock_current |= pd.Series(index >= ep.start, index=index)
    # refreeze hindcast: minimum air from each sample to sunrise
    if len(T):
        fut = T.loc[:sunrise0][::-1].cummin()[::-1]
        out["tmin_to_sunrise"] = fut.reindex(index, method="ffill").values
    else:
        out["tmin_to_sunrise"] = np.nan
    no_freeze = out["fdh"].fillna(0) <= g["fdh_tol"]
    no_refreeze = out["tmin_to_sunrise"].fillna(99) >= g["refreeze_air"]
    rain_ok = (out["since_end_min"] >= g["rain_dry_min"]) & no_freeze & no_refreeze
    snow_ok = (
        no_freeze
        & (out["tmin_since"] >= g["thaw_min_air"])
        & (out["tdh"] >= g["thaw_min_degh"])
        & no_refreeze
        & (out["since_end_min"] >= g["rain_dry_min"])
    )
    out["dry_current"] = ~lock_current & ~out["precip_now"]
    # The thaw rule only relaxes the current policy: it can open a night that the current policy keeps closed
    # (an episode the day crew could not inspect), and never closes a night the current policy allows
    # (an earlier episode is assumed inspected).
    thaw_ok = pd.Series(
        np.where(out["phase"] == "rain", rain_ok, snow_ok), index=index
    ).astype(bool)
    out["thaw_ok"] = thaw_ok & lock_current
    out["dry_thaw"] = (out["dry_current"] | out["thaw_ok"]) & ~out["precip_now"]
    out["dry_thaw"] = out["dry_thaw"].astype(bool)
    return out


import os

ALLSKY_OUT = os.environ.get("ALLSKY_OUT", "/sdf/data/rubin/user/stalder/allsky")


def allsky_cloud(day_obs, every="5min"):
    # allskycam cloud index for one dayObs (rubin_allskycam product), resampled to `every` medians; empty if missing.
    path = os.path.join(ALLSKY_OUT, "cloud", f"{day_obs}.parquet")
    if not os.path.exists(path):
        return pd.Series(dtype=float, name="cloud_idx")
    sky = pd.read_parquet(path)
    if "geometry_ok" in sky:
        sky = sky[sky["geometry_ok"].astype(bool)]
    if len(sky) == 0:
        return pd.Series(dtype=float, name="cloud_idx")
    sky.index = pd.DatetimeIndex(Time(sky["mjd"].to_numpy(), format="mjd").utc.datetime).tz_localize("UTC")
    return sky["cloud_idx"].sort_index().resample(every).median().dropna()


def allsky_deep(day_obs):
    # Per-frame standard and deep allskycam cloud index (UTC index) for one dayObs, joined on seq.
    # ext_deep: grey extinction (mag) that matches the observed V<7.5 star count to the clear-sky model, so it mixes
    # coverage and thickness; the NSIDE-8 cell quartiles of cloud_idx_deep separate a uniform veil (cells bunched at
    # intermediate values) from patchy thick cloud (cells split between ~0 and ~1). Empty if either product is missing.
    p_std = os.path.join(ALLSKY_OUT, "cloud", f"{day_obs}.parquet")
    p_deep = os.path.join(ALLSKY_OUT, "deep_cloud", f"{day_obs}.parquet")
    p_pix = os.path.join(ALLSKY_OUT, "deep_cloud_pix", f"{day_obs}.npz")
    if not (os.path.exists(p_std) and os.path.exists(p_deep)):
        return pd.DataFrame()
    s = pd.read_parquet(p_std)
    s = s[s["geometry_ok"].astype(bool)] if "geometry_ok" in s else s
    d = pd.read_parquet(p_deep)
    m = s[["seq", "mjd", "sun_alt", "moon_alt", "moon_illum", "cloud_idx", "frac_sky_cloudy", "model_key"]].merge(
        d[["seq", "cloud_idx_deep", "ext_deep", "frac_sky_cloudy_deep", "n_cells8"]], on="seq")
    if os.path.exists(p_pix):
        z = np.load(p_pix)
        P = pd.DataFrame(z["cloud_idx_pix8"].astype(float)).set_index(pd.Index(z["seq"], name="seq"))
        q = P.quantile([0.25, 0.5, 0.75], axis=1).T
        q.columns = ["cell_p25", "cell_med", "cell_p75"]
        q["cell_opaque"] = (P >= G["cell_opaque"]).sum(axis=1) / P.notna().sum(axis=1).replace(0, np.nan)
        m = m.merge(q, left_on="seq", right_index=True, how="left")
    m.index = pd.DatetimeIndex(Time(m["mjd"].to_numpy(), format="mjd").utc.datetime).tz_localize("UTC")
    return m.sort_index()


def allsky_ext(day_obs, every="5min"):
    # Deep-index ext_deep (mag) for one dayObs on geometry_ok frames, resampled to `every` medians; plus the stint.
    d = allsky_deep(day_obs)
    if len(d) == 0:
        return pd.Series(dtype=float, name="ext_deep"), ""
    stint = d["model_key"].dropna().str.split("|").str[-1]
    return d["ext_deep"].resample(every).median().dropna(), (stint.mode().iloc[0] if len(stint) else "")


def ext_class(e, bins=None):
    bins = G["ext_bins"] if bins is None else bins
    return pd.cut(e, [-np.inf, *bins, np.inf], labels=["clear", "thin", "moderate", "thick"])


def sky_state_gates(index, cloud, periods, g=G, step_min=1.0, ext=None):
    # Per-sample sky and state gates on `index` (UTC):
    #   cloud_known: an allskycam value within 15 min; clear: rolling-median cloud index <= cloud_clear
    #   down: inside an observatory state listed in down_types (DOWNTIME, FAULT)
    #   ext_known, clear_<name>: with `ext` (deep ext_deep series), rolling-median ext_deep <= g["ext_gates"][name]
    out = pd.DataFrame(index=index)
    if len(cloud):
        c = cloud.reindex(index, method="nearest", tolerance=pd.Timedelta("15min"))
    else:
        c = pd.Series(np.nan, index=index)
    n = max(int(round(g["cloud_window_min"] / step_min)), 1)
    med = c.rolling(n, min_periods=1).median()
    out["cloud_idx"] = c
    out["cloud_known"] = c.notna()
    out["clear"] = (med <= g["cloud_clear"]) & out["cloud_known"]
    e = (ext.reindex(index, method="nearest", tolerance=pd.Timedelta("15min"))
         if ext is not None and len(ext) else pd.Series(np.nan, index=index))
    emed = e.rolling(n, min_periods=1).median()
    out["ext_deep"] = e
    out["ext_known"] = e.notna()
    for name, lim in g["ext_gates"].items():
        out[f"clear_{name}"] = (emed <= lim) & out["ext_known"]
    down = pd.Series(False, index=index)
    if periods is not None and len(periods):
        for _, p_ in periods[periods["type"].isin(g["down_types"])].iterrows():
            s, e = _ts(pd.Timestamp(p_["start"])), _ts(pd.Timestamp(p_["end"]))
            down |= (index >= s) & (index <= e)
    out["down"] = down
    return out



In [ ]:
def night_window(day_obs):
    sunset12, sunrise12 = rn_dayobs.day_obs_sunset_sunrise(day_obs, -12)
    sunset0, sunrise0 = rn_dayobs.day_obs_sunset_sunrise(day_obs, -0.833)
    utc = lambda t: pd.Timestamp(t.utc.datetime).tz_localize("UTC")  # noqa: E731
    return dict(
        sunset0=utc(sunset0),
        sunset12=utc(sunset12),
        sunrise12=utc(sunrise12),
        sunrise0=utc(sunrise0),
    )


def _run_min(ok, minutes, step_min):
    # True where `ok` has been continuously True for at least `minutes`.
    n = max(int(round(minutes / step_min)), 1)
    return ok.astype(float).rolling(n, min_periods=n).min().fillna(0).astype(bool)


def evaluate_rules(w, win, g=G, step_min=1.0, precip_times=None, gates=None):
    # Per-sample rule evaluation inside the night window. `w` is outside_weather() output on a regular grid.
    d = w.loc[(w.index >= win["sunset12"]) & (w.index <= win["sunrise12"])].copy()
    if len(d) == 0:
        return d
    wind_ok = (d["wind_avg5"].fillna(0) < g["wind_avg_close"]) & (
        d["gust"].fillna(0) < g["gust_close"]
    )
    late = d.index > (win["sunrise0"] - pd.Timedelta(hours=g["before_sunrise_h"]))
    dry = pd.Series(True, index=d.index)
    if precip_times is not None and len(precip_times) > 0:
        last_wet = pd.Series(np.nan, index=d.index)
        for t in precip_times:
            last_wet[d.index >= t] = 1.0
        dry = (
            last_wet.isna()
        )  # conservative: any precipitation keeps the rest of the night closed
    dry_thaw = pd.Series(True, index=d.index)
    if (
        gates is not None
    ):  # precipitation gates from precip_gates(); supersedes precip_times
        gg = gates.reindex(d.index, method="ffill")
        dry = gg["dry_current"].fillna(True).astype(bool)
        dry_thaw = gg["dry_thaw"].fillna(True).astype(bool)
    n30 = max(int(round(g["reopen_stable_min"] / step_min)), 1)

    d["table_ok"] = (d["spread"] > g["spread_open"]) & wind_ok & dry
    d["table_close"] = (d["spread_min"] < g["spread_close"]) | ~wind_ok
    d["table_caution"] = (d["spread_min"] <= g["spread_open"]) & ~d["table_close"]
    d["prose_ok"] = (d["RH_max"] <= g["rh_prose"]) & wind_ok & dry
    d["precip_page_ok"] = (d["RH_max"] <= g["rh_precip"]) & wind_ok & dry

    # "May open" = criteria held for the stability window AND not within 1 h of sunrise
    d["open_literal_table"] = (
        _run_min(
            d["table_ok"] & (d["spread_min"] > g["spread_open"]),
            g["reopen_stable_min"],
            step_min,
        )
        & ~late
    )
    d["open_literal_prose"] = (
        _run_min(d["prose_ok"], g["reopen_stable_min"], step_min) & ~late
    )
    d["open_literal_precip"] = _run_min(d["precip_page_ok"], 20, step_min) & ~late
    med = d["spread"].rolling(n30, min_periods=n30).median()
    mn = d["spread_min"].rolling(n30, min_periods=n30).min()
    spread_ok = (med > g["spread_open"]) & (mn > g["spread_close"]) & wind_ok & ~late
    d["open_proposed"] = spread_ok & dry
    d["open_proposed_thaw"] = (
        spread_ok & dry_thaw
    )  # proposed spread rule + proposed thaw rule (RSO-1183)
    d["open_air_only"] = spread_ok  # reference: outside air only, no precipitation gate
    d["spike"] = d["RH_max"] > g["spike_rh"]
    return d


def latch_open(allowed):
    # Once the dome may open, it stays open until a closing limit (handled separately); here: first allowed time onward.
    a = allowed.astype(bool)
    return a.cummax() if a.any() else a


def hours(mask, step_min=1.0):
    return float(mask.sum()) * step_min / 60.0


def segments(mask):
    # Contiguous True runs -> DataFrame(start, end, minutes).
    m = mask.astype(bool)
    if not m.any():
        return pd.DataFrame(columns=["start", "end", "minutes"])
    grp = (m != m.shift()).cumsum()
    out = []
    for _, s in m[m].groupby(grp[m]):
        out.append(
            (
                s.index[0],
                s.index[-1],
                (s.index[-1] - s.index[0]).total_seconds() / 60 + 1,
            )
        )
    return pd.DataFrame(out, columns=["start", "end", "minutes"])

## 1. One night in detail

In [ ]:
win = night_window(DAY_OBS)
t0 = win["sunset0"] - pd.Timedelta(
    hours=4
)  # include the afternoon build-up and handover
t1 = win["sunrise0"] + pd.Timedelta(minutes=30)
print(
    {
        k: to_local(pd.DatetimeIndex([v]))[0].strftime("%Y-%m-%d %H:%M %Z")
        for k, v in win.items()
    }
)

w = outside_weather(t0, t1, "1m")
print(f"outside samples: {len(w)}; columns: {list(w.columns)}")
wet = precipitation(t0, t1)
print(f"precipitation events (raining/snowing True): {len(wet)}")

# Precipitation episodes and thaw bookkeeping over a lookback window (RSO-1183).
t_lb = win["sunset12"] - pd.Timedelta(hours=G["precip_lookback_h"])
wx_lb = outside_weather(t_lb, t1, "5m")
eps = classify_episodes(precip_episodes(t_lb, t1), wx_lb if len(wx_lb) else w)
T_gate = pd.concat([wx_lb["T"], w["T"]]).sort_index() if len(wx_lb) else w["T"]
T_gate = T_gate[~T_gate.index.duplicated(keep="last")]
gates = precip_gates(w.index, eps, T_gate, DAY_OBS, win["sunrise0"])
print(f"precipitation episodes in the last {G['precip_lookback_h']} h: {len(eps)}")
display(w.describe().T.round(2))

In [ ]:
# Sanity check: sensors behind ESS:301 (outside weather station), and its dew point topic vs. the page's formula.
for topic in [
    "lsst.sal.ESS.temperature",
    "lsst.sal.ESS.relativeHumidity",
    "lsst.sal.ESS.dewPoint",
    "lsst.sal.ESS.airFlow",
]:
    s = efd_raw(
        topic,
        ["sensorName", "location"],
        t0,
        t0 + pd.Timedelta(minutes=10),
        index=OUTSIDE_INDEX,
    )
    print(
        topic, sorted(set(s.get("sensorName", []))), sorted(set(s.get("location", [])))
    )
print(
    "median |Td(topic) - Td(formula)| =",
    np.nanmedian(np.abs(w["spread"] - w["spread_formula"])).round(2),
    "C",
)

In [ ]:
r = evaluate_rules(w, win, G, 1.0, gates=gates)
night_h = (win["sunrise12"] - win["sunset12"]).total_seconds() / 3600


def first_time(mask):
    return (
        to_local(mask[mask].index[:1])[0].strftime("%H:%M") if mask.any() else "never"
    )


rows = []
for col, label in [
    ("open_literal_prose", "Prose: RH <= 70% for 30 min"),
    ("open_literal_precip", "Precip page: RH <= 75% for 20 min"),
    ("open_literal_table", "Table, literal: spread > 5 C every minute for 30 min"),
    ("open_proposed", "Proposed: 30-min median spread > 5 C, min > 2 C"),
    ("open_proposed_thaw", "Proposed spread rule + proposed thaw rule"),
    ("open_air_only", "Reference: outside air only (no precipitation gate)"),
]:
    rows.append(
        dict(
            rule=label,
            first_allowed=first_time(r[col]),
            hours_allowed=round(hours(r[col]), 2),
            hours_if_opened_at_first=round(
                hours(latch_open(r[col]) & ~r["table_close"]), 2
            ),
        )
    )
summary = pd.DataFrame(rows)
print(f"Night (-12 to -12): {night_h:.2f} h")
display(summary)

print("Minutes in each table band during the night:")
display(
    pd.Series(
        {
            "can open (spread > 5)": int(r["table_ok"].sum()),
            "caution (2-5)": int(r["table_caution"].sum()),
            "close (< 2 or wind)": int(r["table_close"].sum()),
        }
    )
)

sp = segments(r["spike"])
if len(sp):
    sp["min_spread_C"] = [
        round(float(r.loc[a:b, "spread_min"].min()), 2)
        for a, b in zip(sp.start, sp.end)
    ]
    sp["max_RH"] = [
        round(float(r.loc[a:b, "RH_max"].max()), 1) for a, b in zip(sp.start, sp.end)
    ]
    sp["start"] = to_local(pd.DatetimeIndex(sp.start)).strftime("%H:%M")
    sp["end"] = to_local(pd.DatetimeIndex(sp.end)).strftime("%H:%M")
print(f"RH > {G['spike_rh']}% spikes: {len(sp)}")
display(sp)
print(
    "Would any spike have forced a close had the dome been open?  ambient spread < 2 C minutes:",
    int((r["spread_min"] < G["spread_close"]).sum()),
)

In [ ]:
# Precipitation episodes and the thaw bookkeeping behind the precipitation gates (RSO-1183).
# The current-policy rows (first four rules above) close the rest of the night for any episode ending after
# crew_leave_hour local on the dayObs date; the thaw rule replaces that with the measured criteria in G.
if len(eps):
    ev_tab = eps.copy()
    for c in ["start", "end"]:
        ev_tab[c] = to_local(pd.DatetimeIndex(ev_tab[c])).strftime("%m-%d %H:%M")
    ev_tab["wb_min"] = ev_tab["wb_min"].round(2)
    display(ev_tab)
    gn = gates.loc[(gates.index >= win["sunset12"]) & (gates.index <= win["sunrise12"])]
    marks = {"-12 deg twilight": win["sunset12"]}
    first_thaw = r.index[r["open_proposed_thaw"]]
    if len(first_thaw):
        marks["first allowed (thaw rule)"] = first_thaw[0]
    rows_ = []
    for lab, t in marks.items():
        x = gn.reindex([t], method="nearest").iloc[0]
        rows_.append(
            dict(
                at=lab,
                local=to_local(pd.DatetimeIndex([t]))[0].strftime("%H:%M"),
                phase=x.phase,
                since_end_h=round(x.since_end_min / 60, 2),
                thaw_Ch=round(x.tdh, 1),
                freeze_Ch=round(x.fdh, 2),
                tmin_since_C=round(x.tmin_since, 1),
                tmin_to_sunrise_C=round(x.tmin_to_sunrise, 1),
            )
        )
    display(pd.DataFrame(rows_))
    print(
        f"Hours allowed tonight - current policy (proposed spread rule): {hours(r['open_proposed']):.2f}; "
        f"with thaw rule: {hours(r['open_proposed_thaw']):.2f}; outside air only: {hours(r['open_air_only']):.2f}"
    )
else:
    print(
        f"No precipitation in the {G['precip_lookback_h']} h before the night; the gates are open."
    )

### Inside / structure sensors

Every ESS index that publishes a dew point, compared with that index's first temperature channel. Pairing is per salIndex, so treat this as a screening list and check flagged sensors by name before quoting them. The guidance closes on any sensor within 2 °C of its dew point for about 20 min.

In [ ]:
dp_all = efd_raw(
    "lsst.sal.ESS.dewPoint",
    ["dewPointItem", "sensorName", "location", "salIndex"],
    win["sunset12"],
    win["sunrise12"],
)
sens = []
if len(dp_all):
    for idx, grp in dp_all.groupby("salIndex"):
        td = grp["dewPointItem"].resample("1min").mean()
        tf = (
            first_field("lsst.sal.ESS.temperature", "temperatureItem")
            or "temperatureItem0"
        )
        tt = efd_agg(
            "lsst.sal.ESS.temperature",
            tf,
            win["sunset12"],
            win["sunrise12"],
            "1m",
            "mean",
            idx,
        )
        tt.index = tt.index.floor("1min")
        s = (tt - td.reindex(tt.index)).dropna()
        if len(s) == 0:
            continue
        below = s < G["sensor_close"]
        longest = segments(below)["minutes"].max() if below.any() else 0
        sens.append(
            dict(
                salIndex=int(idx),
                sensor=", ".join(sorted(set(grp.sensorName.astype(str))))[:60],
                location=", ".join(sorted(set(grp.location.astype(str))))[:60],
                min_spread_C=round(s.min(), 2),
                median_spread_C=round(s.median(), 2),
                minutes_below_2C=int(below.sum()),
                longest_run_below_2C_min=float(longest),
            )
        )
sens = pd.DataFrame(sens).sort_values("min_spread_C") if sens else pd.DataFrame()
display(sens)
if len(sens):
    trig = sens[sens.longest_run_below_2C_min >= G["sensor_close_min"]]
    print(
        f"Sensors meeting the 2 C / {G['sensor_close_min']} min close rule: {len(trig)}"
    )

In [ ]:
# Dynalene at mirror level vs. outside dew point ("outside dew point 2 C lower than Dynalene" before opening;
# table: Dynalene - dew point > 0 C can open, below -2 C close).
dyn = (
    efd_agg(
        DYN_TOPIC,
        DYN_FIELD,
        win["sunset12"],
        win["sunrise12"],
        "1m",
        "mean",
        client=cam,
    )
    + DYN_OFFSET
)
if len(dyn):
    m = (dyn - r["Td"].reindex(dyn.index)).dropna()
    print(
        f"Dynalene(camera) - outside Td: min {m.min():.2f} C, median {m.median():.2f} C; "
        f"minutes < 2 C (pre-open check): {(m < 2).sum()}; minutes < -2 C (table close): {(m < -2).sum()}"
    )
else:
    print(
        f"No {DYN_TOPIC}.{DYN_FIELD} in {CAMERA_DB} for this window; last query: {cam.last_query}"
    )

In [ ]:
# Cloud proxy: DIMM and RINGSS measurement cadence (no measurements = no stars, or instrument closed).
# Neither has published since mid-2026 (DIMM last 2026-07-14, RINGSS 2026-06-15), so also read the allskycam
# cloud index (rubin_allskycam products) when they are reachable.
dimm = efd_raw(
    "lsst.sal.DIMM.logevent_dimmMeasurement",
    ["fwhm", "salIndex"],
    win["sunset12"],
    win["sunrise12"],
)
ringss = efd_raw(
    "lsst.sal.ESS.logevent_ringssMeasurement", "*", win["sunset12"], win["sunrise12"]
)
cad = pd.DataFrame(
    {
        "dimm_per_10min": (
            dimm["fwhm"].resample("10min").count()
            if len(dimm)
            else pd.Series(dtype=float)
        ),
        "ringss_per_10min": (
            ringss.iloc[:, 0].resample("10min").count()
            if len(ringss)
            else pd.Series(dtype=float)
        ),
    }
)
if len(dimm):
    print(
        f"DIMM median FWHM while measuring: {dimm['fwhm'].median():.2f} arcsec; first: {to_local(dimm.index[:1])[0]:%H:%M}"
    )
display(cad.set_axis(to_local(cad.index)).T if len(cad) else cad)
import os

ALLSKY_OUT = os.environ.get("ALLSKY_OUT", "/sdf/data/rubin/user/stalder/allsky")
cam_path = os.path.join(ALLSKY_OUT, "cloud", f"{DAY_OBS}.parquet")
sky = pd.DataFrame()
if os.path.exists(cam_path):
    sky = pd.read_parquet(cam_path).query("geometry_ok")
    sky.index = pd.DatetimeIndex(
        Time(sky["mjd"].to_numpy(), format="mjd").utc.datetime
    ).tz_localize("UTC")
    sky = sky.loc[(sky.index >= win["sunset12"]) & (sky.index <= win["sunrise12"])]
    cloud10 = sky["cloud_idx"].resample("10min").median()
    print(
        f"allskycam frames in the night: {len(sky)}; fraction with cloud index > 0.3: {(sky.cloud_idx > 0.3).mean():.2f}, > 0.5: {(sky.cloud_idx > 0.5).mean():.2f}"
    )
    display(
        cloud10.set_axis(to_local(cloud10.index))
        .round(2)
        .to_frame("allsky cloud_idx (10-min median)")
        .T
    )
else:
    print(f"No allskycam cloud table at {cam_path}")

### Cloud thickness: deep allskycam index

The standard index (Yale BSC, V < 5.5) counts stars 1.5–2 mag above the detection limit, so it sees thick cloud and **coverage** but reads thin cloud as clear: 0.5 mag of grey extinction removes about 1 % of those stars. The deep index (Hipparcos V < 7.5, rubin_allskycam `deep_cloud/`) goes to the detection limit and gives `ext_deep`, the grey extinction in magnitudes that matches the observed star count. It is only valid on the sharp stints (FWHM ≈ 1.4 px), which includes `8323/0552@20261016`, the camera in use since 2026-10-16. Columns:

* `cloud_idx` / `frac_sky_cloudy`: standard index and fraction of NSIDE-8 cells > 0.5 (coverage of thick cloud).
* `cloud_idx_deep` / `frac_sky_cloudy_deep`: the same from the deep stars (about 2.5× the cells).
* `ext_deep` (mag): transparency loss. It saturates near 1.5–2 mag; 3.0 means no stars. On clear nights the floor is 0.01–0.05 mag.
* `cell_p25 / cell_med / cell_p75`: spread of the per-cell deep index. Cells bunched at intermediate values mean a **uniform veil**; cells split between ~0 and ~1 mean **patchy thick cloud**. `cell_opaque`: fraction of cells ≥ `cell_opaque`.

Thickness classes on `ext_deep`: clear ≤ 0.2 mag, thin 0.2–0.5, moderate 0.5–1.0, thick > 1.0.

In [ ]:
deep = allsky_deep(DAY_OBS)
if len(deep):
    deep = deep.loc[(deep.index >= win["sunset12"]) & (deep.index <= win["sunrise12"])]
    deep["thickness"] = ext_class(deep["ext_deep"])
    print(f"deep allskycam frames: {len(deep)}; model {deep.model_key.iloc[0]}; "
          f"median ext_deep {deep.ext_deep.median():.2f} mag, p90 {deep.ext_deep.quantile(0.9):.2f} mag")
    step_h = np.median(np.diff(deep.index.asi8)) / 3.6e12
    cls = deep["thickness"].value_counts().reindex(["clear", "thin", "moderate", "thick"]).fillna(0)
    thick_tab = pd.DataFrame({
        "frames": cls.astype(int),
        "hours": (cls * step_h).round(2),
        "std index calls clear (<= cloud_clear)": deep.groupby("thickness", observed=False)["cloud_idx"]
            .apply(lambda x: (x <= G["cloud_clear"]).mean()).round(2),
        "median frac_sky_cloudy_deep": deep.groupby("thickness", observed=False)["frac_sky_cloudy_deep"].median().round(2),
    })
    display(thick_tab)
    cols = ["cloud_idx", "frac_sky_cloudy", "cloud_idx_deep", "frac_sky_cloudy_deep", "ext_deep",
            "cell_p25", "cell_med", "cell_p75", "cell_opaque", "moon_alt"]
    t30 = deep[[c for c in cols if c in deep]].resample("30min").median()
    t30["thickness"] = ext_class(t30["ext_deep"])
    display(t30.set_axis(to_local(t30.index).strftime("%H:%M")).round(2))
    # contiguous stretches by thickness class (on 10-min medians, to avoid frame-to-frame flicker)
    e10 = deep["ext_deep"].resample("10min").median()
    c10 = ext_class(e10).astype(str)
    runs = (c10 != c10.shift()).cumsum()
    seg = pd.DataFrame({
        "start": to_local(pd.DatetimeIndex(e10.index.to_series().groupby(runs).min())).strftime("%H:%M"),
        "end": to_local(pd.DatetimeIndex(e10.index.to_series().groupby(runs).max()) + pd.Timedelta("10min")).strftime("%H:%M"),
        "class": c10.groupby(runs).first(),
        "ext_deep_med_mag": e10.groupby(runs).median().round(2),
        "ext_deep_max_mag": e10.groupby(runs).max().round(2),
        "frac_sky_cloudy_deep_max": deep["frac_sky_cloudy_deep"].resample("10min").median().groupby(runs).max().round(2),
    }).reset_index(drop=True)
    display(seg)
else:
    print(f"No deep allskycam products for {DAY_OBS} under {ALLSKY_OUT}")

In [ ]:
# What the system recorded: dome, observatory state, Watcher, narrative log, scripts.
dome = observatory_status.get_dome_open_close(Time(t0), Time(t1), efd)
display(dome.query("day_obs == @DAY_OBS") if "day_obs" in dome else dome)

try:
    states = observatory_status.get_observatory_state_times(Time(t0), Time(t1), efd)
    display(states)
except Exception as e:  # noqa: BLE001
    print(f"observatory state: {e}")

alarms = efd_raw(
    "lsst.sal.Watcher.logevent_alarm", ['"name"', "severity", "reason"], t0, t1
)  # name is an InfluxQL keyword
pat = re.compile(r"humid|dew|weather|wind|rain|precip|ESS:|dome", re.I)
if len(alarms):
    alarms = alarms[alarms["name"].astype(str).str.contains(pat)].sort_index()
    # one row per episode: severity > 1 events of the same alarm, merged across gaps < 10 min (severity flickers)
    ep = []
    for name, g in alarms[alarms["severity"] > 1].groupby("name"):
        brk = g.index.to_series().diff() > pd.Timedelta(minutes=10)
        for _, e in g.groupby(brk.cumsum()):
            ep.append(
                dict(
                    name=name,
                    start=e.index[0],
                    end=e.index[-1],
                    max_severity=int(e.severity.max()),
                    minutes=round((e.index[-1] - e.index[0]).total_seconds() / 60, 1),
                    first_reason=e.reason.iloc[0][:110],
                )
            )
    alarms = pd.DataFrame(ep)
    if len(alarms):
        alarms = alarms.sort_values("start")
        alarms["start"] = to_local(pd.DatetimeIndex(alarms.start)).strftime(
            "%m-%d %H:%M"
        )
        alarms["end"] = to_local(pd.DatetimeIndex(alarms.end)).strftime("%m-%d %H:%M")
        alarms = alarms.reset_index(drop=True)
display(alarms)

nlog = endpoints["narrative_log"].query_log(Time(t0), Time(t1))
if len(nlog):
    nlog = nlog[~nlog["component"].astype(str).str.contains("AuxTel")]
    nlog.index = to_local(nlog.index)
    cols = [
        c
        for c in ["component", "user_id", "time_lost", "time_lost_type", "message_text"]
        if c in nlog
    ]
    display(nlog[cols])

In [ ]:
# Tonight's allowed time after the sky and state gates: proposed spread rule (current precipitation policy and
# thaw rule) restricted to clear allskycam samples and to time outside DOWNTIME/FAULT.
try:
    states_n = states
except NameError:
    states_n = None
ext_n, stint_n = allsky_ext(DAY_OBS, "5min")
ss = sky_state_gates(r.index, allsky_cloud(DAY_OBS, "5min"), states_n, G, 1.0, ext=ext_n)
print(f"allskycam coverage {ss.cloud_known.mean():.0%}; clear {ss.clear.mean():.0%} of the night; "
      f"DOWNTIME/FAULT {hours(ss.down):.2f} h")
gated = pd.DataFrame({
    "rule": ["Proposed (current precipitation policy)", "Proposed + thaw rule"],
    "allowed_h": [hours(r["open_proposed"]), hours(r["open_proposed_thaw"])],
    "allowed_clear_up_h": [hours(r["open_proposed"] & ss.clear & ~ss.down),
                           hours(r["open_proposed_thaw"] & ss.clear & ~ss.down)],
}).round(2)
# Deep-index transparency gates (30-min rolling median ext_deep <= limit), as in the season scan.
print(f"deep allskycam coverage {ss.ext_known.mean():.0%} (stint {stint_n}, "
      f"{'sharp' if stint_n in G['sharp_stints'] else 'soft: thin cloud under-measured'})")
for name, lim in G["ext_gates"].items():
    gated[f"allowed_{name}_up_h"] = [hours(r["open_proposed"] & ss[f"clear_{name}"] & ~ss.down),
                                     hours(r["open_proposed_thaw"] & ss[f"clear_{name}"] & ~ss.down)]
    sg = segments(r["open_proposed_thaw"] & ss[f"clear_{name}"] & ~ss.down)
    if len(sg):
        sg["start"] = to_local(pd.DatetimeIndex(sg.start)).strftime("%H:%M")
        sg["end"] = to_local(pd.DatetimeIndex(sg.end)).strftime("%H:%M")
    print(f"Proposed + thaw rule, ext_deep <= {lim} mag ({name}) and not DOWNTIME/FAULT: "
          + (", ".join(f"{x.start}-{x.end}" for x in sg.itertuples()) if len(sg) else "none"))
gated = gated.round(2)
display(gated)

In [ ]:
try:
    import rubin_nights.scriptqueue as scriptqueue

    scripts = scriptqueue.get_script_stream(Time(t0), Time(t1), efd)
    scripts.index = (
        to_local(pd.DatetimeIndex(scripts.index))
        if isinstance(scripts.index, pd.DatetimeIndex)
        else scripts.index
    )
    keep = [
        c
        for c in scripts.columns
        if re.search(r"script|block|config|salIndex", c, re.I)
    ][:6]
    display(scripts[keep] if keep else scripts.head(50))
except Exception as e:  # noqa: BLE001
    print(f"script stream: {e}")

In [ ]:
# Figure: one night against the limits.
L = lambda s: s.set_axis(to_local(s.index))  # noqa: E731
fig, ax = plt.subplots(
    5,
    1,
    figsize=(11, 12),
    sharex=True,
    gridspec_kw=dict(height_ratios=[2, 2, 1.3, 1.5, 1.1]),
)
xw = L(w)

ax[0].plot(xw.index, xw["T"], label="T (ESS:301)")
ax[0].plot(xw.index, xw["Td"], label="dew point (ESS:301)")
if len(dyn):
    ax[0].plot(L(dyn).index, L(dyn), label="Dynalene (camera supply)")
ax[0].set_ylabel("°C")
ax[0].legend(loc="upper right", fontsize=8)

ax[1].plot(xw.index, xw["spread"], color="k", lw=1, label="spread (1-min mean)")
ax[1].plot(
    xw.index, xw["spread_min"], color="0.6", lw=0.8, label="spread (1-min worst)"
)
ax[1].axhspan(-5, G["spread_close"], color="tab:red", alpha=0.12)
ax[1].axhspan(G["spread_close"], G["spread_open"], color="tab:orange", alpha=0.12)
ax[1].set_ylim(bottom=max(-1, np.nanmin(xw["spread_min"]) - 1))
ax[1].set_ylabel("T − T_dew (°C)")
a2 = ax[1].twinx()
a2.plot(xw.index, xw["RH"], color="tab:blue", lw=0.8, alpha=0.7)
for v, ls in [(G["rh_prose"], ":"), (G["rh_precip"], "--"), (G["spike_rh"], "-")]:
    a2.axhline(v, color="tab:blue", ls=ls, lw=0.8)
a2.set_ylabel("RH (%)", color="tab:blue")
ax[1].legend(loc="upper left", fontsize=8)

ax[2].plot(xw.index, xw["wind_avg5"], label="wind, 5-min avg")
ax[2].plot(xw.index, xw["gust"], lw=0.6, label="gust")
ax[2].axhline(G["wind_avg_close"], color="tab:red", ls="--", lw=0.8)
ax[2].axhline(G["gust_close"], color="tab:red", lw=0.8)
ax[2].set_ylabel("m/s")
ax[2].legend(loc="upper right", fontsize=8)

if len(deep):
    xd = L(deep)
    ax[3].plot(xd.index, xd["ext_deep"], color="k", lw=0.9, label="ext_deep (mag, deep index)")
    ax[3].plot(xd.index, xd["frac_sky_cloudy_deep"], color="tab:purple", lw=0.8, label="cells cloudy (deep)")
    ax[3].plot(xd.index, xd["cloud_idx"], color="tab:orange", lw=0.8, label="cloud_idx (standard)")
    for v in G["ext_bins"]:
        ax[3].axhline(v, color="0.5", ls=":", lw=0.7)
    ax[3].axhline(G["cloud_clear"], color="tab:orange", ls="--", lw=0.7)
    ax[3].set_ylim(0, max(1.2, min(3.0, float(np.nanmax(xd["ext_deep"])) + 0.1)))
ax[3].set_ylabel("allskycam")
ax[3].legend(loc="upper right", fontsize=8)

lanes = [
    ("open_proposed_thaw", "proposed + thaw"),
    ("open_proposed", "proposed"),
    ("open_literal_table", "table, literal"),
    ("open_literal_precip", "precip page"),
    ("open_literal_prose", "prose RH 70%"),
]
for i, (col, lab) in enumerate(lanes):
    m = L(r[col])
    for _, sgm in segments(m).iterrows():
        ax[4].broken_barh(
            [(mdates.date2num(sgm.start), (sgm.minutes) / 1440)],
            (i - 0.35, 0.7),
            color="tab:green",
        )
    ax[4].text(
        1.002, i, lab, transform=ax[4].get_yaxis_transform(), va="center", fontsize=8
    )
if "open_time" in dome:
    for _, dr in dome.query("day_obs == @DAY_OBS").iterrows():
        if pd.notna(dr.open_time):
            o = to_local(pd.DatetimeIndex([dr.open_time]))[0]
            c = (
                to_local(pd.DatetimeIndex([dr.close_time]))[0]
                if pd.notna(dr.close_time)
                else xw.index[-1]
            )
            ax[4].broken_barh(
                [(mdates.date2num(o), mdates.date2num(c) - mdates.date2num(o))],
                (len(lanes) - 0.35, 0.7),
                color="k",
            )
ax[4].text(
    1.002,
    len(lanes),
    "dome actually open",
    transform=ax[4].get_yaxis_transform(),
    va="center",
    fontsize=8,
)
ax[4].set_yticks([])
ax[4].set_ylim(-0.6, len(lanes) + 0.6)
ax[4].set_title("When each rule would allow opening (green)", fontsize=9, loc="left")

for a in ax:
    for k in ["sunset12", "sunrise12"]:
        a.axvline(to_local(pd.DatetimeIndex([win[k]]))[0], color="0.3", ls=":", lw=0.8)
    if len(nlog):
        for t in nlog.index:
            a.axvline(t, color="tab:purple", alpha=0.25, lw=0.6)
ax[-1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=SITE_TZ))
fig.suptitle(
    f"dayObs {DAY_OBS}: outside conditions vs. Simonyi open/close guidance (purple = narrative log)"
)
fig.tight_layout()
plt.show()

## 2. Commissioning through operations

Each night is scored on a 5-min grid against the same rules. For each night, `allowed_*` is the hours the rule would have permitted opening, and `open_h` is the hours the dome was actually open. These are categories to triage, not verdicts:

* **possible_missed:** an available night (not in `UNAVAILABLE`) with the dome closed (open_h < 0.25) while the proposed rule allowed ≥ 2 h. Check the narrative log and the closure list: engineering, faults, staffing or road closures may explain it.
* **open_past_limit:** minutes the dome was open while the ambient spread was < 2 °C or wind was over a limit.

Precipitation is gated with the current-policy proxy (`allowed_*`) and with the proposed thaw rule (`allowed_proposed_thaw_h`), which only relaxes the proxy: it can open a night the proxy keeps closed, never the reverse, so `thaw_gain_h` ≥ 0; `allowed_air_only_h` ignores precipitation. Clouds and forecasts are not modeled (the refreeze test uses the measured temperature to sunrise as a stand-in for the forecast), so this scan finds candidates only. Weekend observer inspections are not modeled in the current-policy proxy.

**Sky and state gates.** `allowed_clear_h` and `allowed_thaw_clear_h` keep only time when the 30-min median allskycam cloud index is ≤ `cloud_clear` and the observatory is not in DOWNTIME or FAULT. `possible_missed_clear` uses them and needs allskycam coverage of at least `cloud_min_coverage` of the night. Engineering nights logged as OPERATIONAL are not caught by the state gate; add confirmed blocks to `UNAVAILABLE`.

**Deep-index transparency gates.** `allowed_ext1_h` / `allowed_thaw_ext1_h` keep time when the 30-min median deep `ext_deep` is ≤ 1 mag (proposed: observe through thin and moderate cloud). `allowed_ext02_h` / `allowed_thaw_ext02_h` use ≤ 0.2 mag, likely aligned with current practice (clear sky only). Both also drop DOWNTIME/FAULT. `possible_missed_ext1` / `possible_missed_ext02` need deep coverage ≥ `cloud_min_coverage` (`screened_deep`). `deep_stint` and `deep_sharp` record the allskycam configuration. `ext_deep` measures thin cloud only on the sharp stints (FWHM ≈ 1.4 px: `8323/0552@20261016`, i.e. from 2026-10-16). Most of this season is on the soft stints (8323/0552@20231118, 9118/0165@20251204), where the deep stars barely go deeper than the standard index. There, the 0.2 mag gate passes some thin cloud, so it gives an upper bound on clear time.

In [ ]:
def season_scan(day_min, day_max, every=SEASON_BIN):
    step = pd.Timedelta(every).total_seconds() / 60
    t_lo = rn_dayobs.day_obs_to_time(day_min)
    t_hi = rn_dayobs.day_obs_to_time(day_max) + TimeDelta(1, format="jd")
    chunks, t = [], t_lo
    while t < t_hi:
        tn = min(t + TimeDelta(14, format="jd"), t_hi)
        chunks.append(outside_weather(t, tn, every))
        t = tn
    W = pd.concat([c for c in chunks if len(c)]).sort_index()
    W = W[~W.index.duplicated()]
    W["wind_avg5"] = W["wind"]  # at 5-min binning the mean is already the 5-min average
    t_lb = _ts(t_lo) - pd.Timedelta(hours=G["precip_lookback_h"])
    E_all, t = [], Time(t_lb.to_pydatetime())
    while t < t_hi:
        tn = min(t + TimeDelta(14, format="jd"), t_hi)
        E_all.append(precip_episodes(t, tn))
        t = tn
    E_all = (
        pd.concat([e for e in E_all if len(e)])
        if any(len(e) for e in E_all)
        else pd.DataFrame(columns=["start", "end", "snow_flag"])
    )
    E_all = (
        classify_episodes(E_all.sort_values("start").reset_index(drop=True), W)
        if len(E_all)
        else classify_episodes(E_all, W)
    )
    dome_all = observatory_status.get_dome_open_close(t_lo, t_hi, efd)
    try:
        states_all = observatory_status.get_observatory_state_times(t_lo, t_hi, efd)
    except Exception as e:  # noqa: BLE001
        print(f"observatory states unavailable: {type(e).__name__}: {e}")
        states_all = pd.DataFrame(columns=["day_obs", "start", "end", "hours", "type"])
    nlog_all = endpoints["narrative_log"].query_log(
        t_lo, t_hi, {"min_time_lost": "0.00001"}
    )
    if len(nlog_all):
        nlog_all = nlog_all[~nlog_all["component"].astype(str).str.contains("AuxTel")]
    out = []
    for day in rn_dayobs.day_obs_list(t_lo, t_hi - TimeDelta(1, format="jd")):
        day = int(str(day).replace("-", ""))
        wn = night_window(day)
        nidx = W.index[(W.index >= wn["sunset12"]) & (W.index <= wn["sunrise12"])]
        lb0 = wn["sunset12"] - pd.Timedelta(hours=G["precip_lookback_h"])
        En = (
            E_all[(E_all.end >= lb0) & (E_all.start <= wn["sunrise12"])]
            if len(E_all)
            else E_all
        )
        gt = (
            precip_gates(
                nidx, En, W["T"].loc[lb0 : wn["sunrise0"]], day, wn["sunrise0"]
            )
            if len(nidx)
            else None
        )
        rr = evaluate_rules(W, wn, G, step, gates=gt)
        if len(rr) == 0:
            continue
        ext_d, stint_d = allsky_ext(day, SEASON_BIN.replace("m", "min"))
        ss = sky_state_gates(rr.index, allsky_cloud(day, SEASON_BIN.replace("m", "min")), states_all, G, step,
                             ext=ext_d)
        up_clear = ss["clear"] & ~ss["down"]
        ext_cols = {}
        for name in G["ext_gates"]:
            up_e = ss[f"clear_{name}"] & ~ss["down"]
            ext_cols[f"allowed_{name}_h"] = round(hours(rr["open_proposed"] & up_e, step), 2)
            ext_cols[f"allowed_thaw_{name}_h"] = round(hours(rr["open_proposed_thaw"] & up_e, step), 2)
            ext_cols[f"{name}_frac"] = round(float(ss[f"clear_{name}"].mean()), 2)
        dd = (
            dome_all.query("day_obs == @day")
            if "day_obs" in dome_all
            else pd.DataFrame()
        )
        open_mask = pd.Series(False, index=rr.index)
        for _, x in dd.iterrows():
            if pd.notna(x.get("open_time")):
                o = pd.Timestamp(x.open_time)
                o = o.tz_localize("UTC") if o.tzinfo is None else o
                c = (
                    pd.Timestamp(x.close_time)
                    if pd.notna(x.get("close_time"))
                    else rr.index[-1]
                )
                c = c.tz_localize("UTC") if c.tzinfo is None else c
                open_mask |= (rr.index >= o) & (rr.index <= c)
        lost = (
            nlog_all[
                (nlog_all.index >= wn["sunset12"] - pd.Timedelta(hours=3))
                & (nlog_all.index <= wn["sunrise12"] + pd.Timedelta(hours=3))
            ]
            if len(nlog_all)
            else pd.DataFrame()
        )
        out.append(
            dict(
                day_obs=day,
                night_h=round(
                    (wn["sunrise12"] - wn["sunset12"]).total_seconds() / 3600, 2
                ),
                open_h=round(hours(open_mask, step), 2),
                allowed_prose_h=round(hours(rr["open_literal_prose"], step), 2),
                allowed_table_literal_h=round(hours(rr["open_literal_table"], step), 2),
                allowed_proposed_h=round(hours(rr["open_proposed"], step), 2),
                allowed_proposed_thaw_h=round(hours(rr["open_proposed_thaw"], step), 2),
                allowed_air_only_h=round(hours(rr["open_air_only"], step), 2),
                allowed_clear_h=round(hours(rr["open_proposed"] & up_clear, step), 2),
                allowed_thaw_clear_h=round(hours(rr["open_proposed_thaw"] & up_clear, step), 2),
                cloud_coverage=round(float(ss["cloud_known"].mean()), 2),
                clear_frac=round(float(ss["clear"].mean()), 2),
                deep_coverage=round(float(ss["ext_known"].mean()), 2),
                deep_stint=stint_d,
                deep_sharp=stint_d in G["sharp_stints"],
                ext_deep_med=round(float(ss["ext_deep"].median()), 2) if ss["ext_known"].any() else np.nan,
                **ext_cols,
                down_h=round(hours(ss["down"], step), 2),
                precip_episodes=int(len(En)),
                precip_phase=",".join(sorted(set(En.phase))) if len(En) else "",
                open_past_limit_min=int((open_mask & rr["table_close"]).sum() * step),
                spikes=int(len(segments(rr["spike"]))),
                min_spread_C=round(float(rr["spread_min"].min()), 1),
                max_gust=round(float(rr["gust"].max()), 1),
                log_weather_h=(
                    round(
                        float(
                            lost.query("time_lost_type == 'weather'")["time_lost"].sum()
                        ),
                        2,
                    )
                    if len(lost)
                    else 0.0
                ),
                log_fault_h=(
                    round(
                        float(
                            lost.query("time_lost_type == 'fault'")["time_lost"].sum()
                        ),
                        2,
                    )
                    if len(lost)
                    else 0.0
                ),
            )
        )
    S = pd.DataFrame(out).set_index("day_obs")
    S["available"] = True
    S["unavailable_reason"] = ""
    for d0, d1, why in UNAVAILABLE:
        sel = (S.index >= d0) & (S.index <= d1)
        S.loc[sel, "available"] = False
        S.loc[sel, "unavailable_reason"] = why
    S["possible_missed"] = S.available & (S.open_h < 0.25) & (S.allowed_proposed_h >= 2)
    # Screened: also needs allskycam coverage, and >= 2 h allowed while clear and outside DOWNTIME/FAULT.
    S["screened"] = S.available & (S.cloud_coverage >= G["cloud_min_coverage"])
    S["possible_missed_clear"] = S.screened & (S.open_h < 0.25) & (S.allowed_clear_h >= 2)
    # Deep-index screens: same as possible_missed_clear with the ext_deep transparency gates.
    S["screened_deep"] = S.available & (S.deep_coverage >= G["cloud_min_coverage"])
    for name in G["ext_gates"]:
        S[f"possible_missed_{name}"] = S.screened_deep & (S.open_h < 0.25) & (S[f"allowed_{name}_h"] >= 2)
    S["thaw_clear_gain_h"] = (S.allowed_thaw_clear_h - S.allowed_clear_h).round(2)
    S["prose_vs_proposed_h"] = (S.allowed_proposed_h - S.allowed_prose_h).round(2)
    S["thaw_gain_h"] = (S.allowed_proposed_thaw_h - S.allowed_proposed_h).round(2)
    return S


if RUN_SEASON:
    scan_params = (SEASON_MIN, SEASON_MAX, SEASON_BIN, OUTSIDE_INDEX, tuple(G["ext_gates"].items()))
    if "S" not in globals() or globals().get("_SEASON_SCAN_PARAMS") != scan_params:
        S = season_scan(SEASON_MIN, SEASON_MAX)
        _SEASON_SCAN_PARAMS = scan_params
    S.to_csv(f"dome_guidance_season_{SEASON_MIN}_{SEASON_MAX}.csv")
    print(
        f"{len(S)} nights ({int(S.available.sum())} available); possible_missed: {int(S.possible_missed.sum())}; "
        f"nights open past a limit: {int((S.open_past_limit_min > 0).sum())}"
    )
    print(
        f"Hours allowed, summed: prose RH 70% = {S.allowed_prose_h.sum():.0f}, "
        f"table literal = {S.allowed_table_literal_h.sum():.0f}, "
        f"current-policy proposed = {S.allowed_proposed_h.sum():.0f}, "
        f"proposed thaw = {S.allowed_proposed_thaw_h.sum():.0f}, "
        f"outside air only = {S.allowed_air_only_h.sum():.0f}; "
        f"actual open = {S.open_h.sum():.0f}"
    )
    display(S.query("possible_missed or open_past_limit_min > 0"))
    print("Nights where the thaw rule adds open time over the current-policy proxy:")
    display(
        S.query("thaw_gain_h > 0")[
            [
                "open_h",
                "allowed_proposed_h",
                "allowed_proposed_thaw_h",
                "thaw_gain_h",
                "precip_phase",
            ]
        ]
    )

In [ ]:
if RUN_SEASON:
    if "S" in globals() and "_SEASON_SCAN_PARAMS" not in globals():
        _SEASON_SCAN_PARAMS = (SEASON_MIN, SEASON_MAX, SEASON_BIN, OUTSIDE_INDEX, tuple(G["ext_gates"].items()))

    print(
        f"{len(S)} nights ({int(S.available.sum())} available); possible_missed: {int(S.possible_missed.sum())}; "
        f"nights open past a limit: {int((S.open_past_limit_min > 0).sum())}"
    )
    A = S[S.available]
    print(
        f"Available nights only - hours allowed: current-policy proposed = {A.allowed_proposed_h.sum():.0f}, "
        f"proposed thaw = {A.allowed_proposed_thaw_h.sum():.0f}, outside air only = {A.allowed_air_only_h.sum():.0f}; "
        f"actual open = {A.open_h.sum():.0f}"
    )
    print(
        f"Hours allowed, summed: prose RH 70% = {S.allowed_prose_h.sum():.0f}, "
        f"table literal = {S.allowed_table_literal_h.sum():.0f}, "
        f"current-policy proposed = {S.allowed_proposed_h.sum():.0f}, "
        f"proposed thaw = {S.allowed_proposed_thaw_h.sum():.0f}, "
        f"outside air only = {S.allowed_air_only_h.sum():.0f}; "
        f"actual open = {S.open_h.sum():.0f}"
    )
    gain = S[S.thaw_gain_h > 0]
    loss = S[S.thaw_gain_h < 0]
    print(
        f"Thaw rule adds allowed time on {len(gain)} nights ({gain.thaw_gain_h.sum():.1f} h total), "
        f"reduces it on {len(loss)} nights ({loss.thaw_gain_h.sum():.1f} h net); "
        f"net change {S.thaw_gain_h.sum():.1f} h."
    )
    print("Hours and nights by precip phase in the 72-h lookback:")
    display(
        S.groupby("precip_phase")
        .agg(
            nights=("open_h", "size"),
            actual_open_h=("open_h", "sum"),
            current_policy_h=("allowed_proposed_h", "sum"),
            thaw_rule_h=("allowed_proposed_thaw_h", "sum"),
            thaw_delta_h=("thaw_gain_h", "sum"),
        )
        .round(1)
    )
    print("Largest positive thaw-rule gains:")
    display(
        gain.sort_values("thaw_gain_h", ascending=False).head(12)[
            [
                "open_h",
                "allowed_proposed_h",
                "allowed_proposed_thaw_h",
                "thaw_gain_h",
                "precip_phase",
            ]
        ]
    )
    print("Possible missed nights (current-policy screen), ranked by allowed time:")
    display(
        S.query("possible_missed")
        .sort_values("allowed_proposed_h", ascending=False)
        .head(12)[
            [
                "open_h",
                "allowed_proposed_h",
                "allowed_proposed_thaw_h",
                "thaw_gain_h",
                "precip_phase",
                "log_weather_h",
                "log_fault_h",
            ]
        ]
    )
    print("Night(s) open during a modeled closing-limit interval:")
    display(
        S.query("open_past_limit_min > 0")[
            ["open_h", "open_past_limit_min", "min_spread_C", "max_gust"]
        ]
    )

    m = S.copy()
    m.index = pd.to_datetime(m.index.astype(str))
    series = [
        ("allowed_proposed_h", "current-policy proposed"),
        ("allowed_proposed_thaw_h", "proposed thaw"),
        ("allowed_air_only_h", "outside air only"),
        ("allowed_prose_h", "RH 70% prose"),
        ("allowed_table_literal_h", "table literal"),
        ("open_h", "dome actually open"),
    ]
    mo = m.resample("MS")[[column for column, _ in series]].sum()
    fig, ax = plt.subplots(figsize=(13, 4))
    x = np.arange(len(mo))
    width = 0.13
    for i, (column, label) in enumerate(series):
        offset = (i - (len(series) - 1) / 2) * width
        ax.bar(
            x + offset,
            mo[column],
            width,
            label=label,
            color="k" if column == "open_h" else None,
        )
    ax.set_xticks(x)
    ax.set_xticklabels(mo.index.strftime("%Y-%m"), rotation=60)
    ax.set_ylabel("hours (−12° to −12°)")
    ax.legend(fontsize=8, ncol=3)
    ax.set_title("Night hours allowed by each rule vs. hours open, by month")
    fig.tight_layout()
    plt.show()

    # Sky and state gates (allskycam cloud index, DOWNTIME/FAULT): what explains the ungated screen?
    if "possible_missed_clear" in S:
        pm = S[S.possible_missed]
        why = pd.Series("still flagged", index=pm.index)
        why[pm.down_h >= 0.5 * pm.night_h] = "DOWNTIME/FAULT most of the night"
        why[(why == "still flagged") & (pm.cloud_coverage < G["cloud_min_coverage"])] = "no allskycam data"
        why[(why == "still flagged") & (pm.allowed_clear_h < 2)] = "cloudy (allowed < 2 h while clear)"
        print("Ungated possible-missed nights, by what the gates find:")
        display(why.value_counts().to_frame("nights"))
        A = S[S.available]
        print(
            f"Available nights with allskycam coverage: {int(S.screened.sum())} of {len(A)}. "
            f"Clear-sky, not-down allowed hours: current policy {A.allowed_clear_h.sum():.0f}, "
            f"thaw rule {A.allowed_thaw_clear_h.sum():.0f}; thaw gain {A.thaw_clear_gain_h.sum():.1f} h "
            f"on {int((A.thaw_clear_gain_h > 0).sum())} nights."
        )
        print(f"Possible missed after the gates: {int(S.possible_missed_clear.sum())} nights")
        display(
            S.query("possible_missed_clear").sort_values("allowed_clear_h", ascending=False)[
                ["open_h", "allowed_proposed_h", "allowed_clear_h", "clear_frac", "down_h",
                 "precip_phase", "log_weather_h", "log_fault_h"]
            ]
        )

    # Deep allskycam transparency gates: ext_deep <= 1 mag (proposed) and <= 0.2 mag (likely current guidance).
    if "screened_deep" in S:
        A = S[S.available & S.screened_deep]
        print(f"\nDeep-index gates on {len(A)} available nights with deep coverage "
              f"({int(A.deep_sharp.sum())} on a sharp stint, where ext_deep sees thin cloud; on the soft stints "
              f"it under-measures thin cloud, so the 0.2 mag gate is permissive there):")
        cols = ["allowed_clear_h"] + [f"allowed_{n}_h" for n in G["ext_gates"]] + \
               ["allowed_thaw_clear_h"] + [f"allowed_thaw_{n}_h" for n in G["ext_gates"]]
        tab = A.groupby("deep_stint")[cols + ["open_h"]].sum().round(0)
        tab.insert(0, "nights", A.groupby("deep_stint").size())
        tab.loc["all"] = tab.sum()
        display(tab)
        lbl = {n: f"ext_deep <= {lim} mag" for n, lim in G["ext_gates"].items()}
        print("Possible missed nights (closed, >= 2 h allowed and not down) under each sky gate:")
        display(pd.Series({"standard index <= cloud_clear": int(S.possible_missed_clear.sum()),
                           **{lbl[n]: int(S[f"possible_missed_{n}"].sum()) for n in G["ext_gates"]}}).to_frame("nights"))
        cols_pm = ["open_h", "allowed_proposed_h", "allowed_clear_h"] + [f"allowed_{n}_h" for n in G["ext_gates"]] + \
                  ["ext_deep_med", "deep_stint", "precip_phase", "log_weather_h", "log_fault_h"]
        first = next(iter(G["ext_gates"]))
        display(S.query(f"possible_missed_{first}").sort_values(f"allowed_{first}_h", ascending=False)[cols_pm])

        mo = A.copy()
        mo.index = pd.to_datetime(mo.index.astype(str))
        ser = [("allowed_proposed_h", "proposed (no sky gate)"), ("allowed_clear_h", "standard index clear")] + \
              [(f"allowed_{n}_h", lbl[n]) for n in G["ext_gates"]] + [("open_h", "dome actually open")]
        mo = mo.resample("MS")[[c for c, _ in ser]].sum()
        fig, ax = plt.subplots(figsize=(13, 4))
        x = np.arange(len(mo)); wd = 0.16
        for i, (c, l) in enumerate(ser):
            ax.bar(x + (i - (len(ser) - 1) / 2) * wd, mo[c], wd, label=l, color="k" if c == "open_h" else None)
        ax.set_xticks(x); ax.set_xticklabels(mo.index.strftime("%Y-%m"), rotation=60)
        ax.set_ylabel("hours (−12° to −12°)")
        ax.legend(fontsize=8, ncol=3)
        ax.set_title("Proposed-rule hours under the allskycam sky gates (available nights with deep coverage), by month")
        fig.tight_layout()
        plt.show()
